# Problem 009:  Special Pythagorean Triplet

> A Pythagorean triplet is a set of three natural numbers, $a \lt b \lt c$, for which,
>
> $$a^2 + b^2 = c^2.$$
>
> For example, $3^2 + 4^2 = 9 + 16 = 25 = 5^2$.
>
> There exists exactly one Pythagorean triplet for which $a + b + c = 1000$.
> Find the product $abc$.


## Naive algorithm $\mathcal O (N^2)$

As with many of these first 100 problems, an inefficient code is more than enough to calculate the answer.  Here, a double loop of $a < b < N$ is done, where $N$ is the sum of the Pythagorean Triplet.  A few changes can be used to speed up the search.  The smallest value, $a$, must be less than $N/3$, otherwise the sum $a+b+c$ would be more than the target value $N$.  Likewise, the middle value, $b$, must be less than $(N - a) / 2$ to ensure the sum $a+b+c$ is less than or equal to the target value $N$.  The code stops as soon as it encounters a triplet that sums to $N$, or runs over all possible values and returns 0.

The time complexity is $\mathcal O(N^2)$, due to the double loop.

In [20]:
%%time

def p009_naive(N: int) -> int:
    for a in range(1, N // 3):
        bmax = (N - a) // 2
        for b in range(a+1, bmax):
            c = N - a - b
            if a**2 + b**2 == c**2:
                return a * b * (N - a - b)
    return 0

N = 1_000
ans = p009_naive(N)
print(ans)

31875000
CPU times: user 26.1 ms, sys: 879 μs, total: 26.9 ms
Wall time: 26.2 ms


## Faster approach $\mathcal O (\sqrt N \ln N)$

Pythagorean Triplets can be generated directly using auxillary variables $n$, $m$, and $k$.  Namely,
$$a = k(m^2 - n^2); b = 2knm; c = k(m^2 + n^2).$$
Given $m>n$, coprime, and one is odd, the other even, all Pythagorean triplets can be generated.  So the problem at hand is to find values of $m$, $n$, and $k$ s.t.
$$N = a + b + c = 2km(m+n).$$
The challenge is to factor $N/2$ in such a way that there are two factors $m$ and $n+m$ s.t. $m>n$ and $n+m$ is odd.  A smart, but cumbersome, code would involve finding the prime factorization of $N/2$ and searching for such a factorization.  Here, I do trial division of all $i < \sqrt(N/2)$ followed by trial division of odd $i + 1 < j < 2i$.  Any values that satisfy those two properties produce a Pythagorean triplet desired.  The extra stipulation that $m$ and $n$ are coprime can be disregarded; all it does is finds a unique $n$ and $m$ for a given Pythagorean triplet.  Since the problem is asking for a single representation, finding $m$ and $n$ coprime will only slow the code.  But, if the question was to find the number of unique Pythagorean triplets with the property, the coprime detail would become relavant again.

As for time scaling, the $\mathcal O(\sqrt N\ln N)$ is definitely an upper bound. The motivation is that the average number of divisors for all integers below $N$ is $\ln N$.  Then, for each divisor found, a loop of size $\sim \sqrt N$ is performed to find the second factors.  In the worse case scenario, there are no Pythagoread triplets that sum to the target and the code will do the complete loops.  Emperically, the code does not scale in that way.  Putting a highly composite number that's 20 digits long runs just as fast as 1000.  The reason for this is because if the number is highly composite then there will be more chances to find a factorization of the desired type.


In [32]:
%%time

from math import gcd

def p009(N: int) -> int:
    if N % 2:
        return 0
    N //= 2
    i = 1
    while i * i <= N:
        if N % i == 0:
            Np = N // i
            j = ((i + 1) // 2) * 2 + 1
            while j < 2 * i:
                #if Np % j == 0 and gcd(i, j) == 1:
                if Np % j == 0:
                    #print(i,j, Np//j)
                    a = (Np // j) * (i**2 - (j - i)**2)
                    b = (Np // j) * (2 * i * (j - i))
                    c = (Np // j) * (i**2 + (j - i)**2)
                    return a*b*c
                j += 2
        i += 1
    return 0

N = 1_000
ans = p009(N)
print(ans)

31875000
CPU times: user 228 μs, sys: 0 ns, total: 228 μs
Wall time: 216 μs
